In [28]:
import langchain
import os 
from dotenv import load_dotenv


In [29]:

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [30]:
from langchain.chat_models import init_chat_model

# tool

In [39]:

model = init_chat_model("llama-3.1-8b-instant", model_provider="groq")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x74b8f3118080>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x74b8f3118fb0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [40]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """ Get the weather at a location """
    return f"Its sunny/cold in {location}"

model_with_tools=model.bind_tools([get_weather])

In [41]:
response=model_with_tools.invoke("whats the weather in noida")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': '68reky02s', 'function': {'arguments': '{"location":"noida"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 220, 'total_tokens': 236, 'completion_time': 0.032338838, 'completion_tokens_details': None, 'prompt_time': 0.149369155, 'prompt_tokens_details': None, 'queue_time': 0.052091542, 'total_time': 0.181707993}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019db97a-6999-78b0-a53a-14245b3dd75d-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'noida'}, 'id': '68reky02s', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 16, 'total_tokens': 236}


In [42]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'noida'},
  'id': '68reky02s',
  'type': 'tool_call'}]

In [43]:
from langchain_core.messages import HumanMessage, ToolMessage

# step 1 : Model generates tool calls
query = "whats the weather in noida"
messages = [HumanMessage(content=query)] # changed to HumanMessage

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# step 2 Execute tools (with generated arguments)
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3 : Final response from model using tool results
final_response = model_with_tools.invoke(messages)

# Final print (changed .text to .content)
print(final_response.content)


Note: The actual weather may vary based on the current time and date. The given information is based on the available data.


In [27]:
messages

[HumanMessage(content='whats the weather in noida', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '82bq8tf22', 'function': {'arguments': '{"location":"noida"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 220, 'total_tokens': 236, 'completion_time': 0.027999185, 'completion_tokens_details': None, 'prompt_time': 0.013536084, 'prompt_tokens_details': None, 'queue_time': 0.05194457, 'total_time': 0.041535269}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019db96f-2163-7020-8968-8479e0929140-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'noida'}, 'id': '82bq8tf22', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 16, 'total_tokens': 236})